## Trabalho de Conclusão de Curso – Ciência da Computação  
**Instituição:** Centro Universitário Carioca (UniCarioca)  
**Alunos:** Daniel de Sousa Santiago, Luís Felipe Vaz
**Orientador:** Prof. Dr. Ricardo Mesquita

        

### Contextualização

Este notebook apresenta a implementação prática do algoritmo *k-Nearest Neighbors Regressor (k-NN Regressor)*, utilizada como suporte experimental para o Trabalho de Conclusão de Curso. O objetivo é demonstrar, de forma reprodutível, as etapas de pré-processamento, configuração do modelo, avaliação de desempenho e realização de previsões.


### Objetivo do Experimento

O objetivo deste experimento é avaliar o desempenho do algoritmo *k-NN Regressor* em um problema de regressão supervisionada, analisando o impacto do pré-processamento dos dados e da escolha do número de vizinhos (*k*) na qualidade das previsões.


### Conjunto de Dados

O conjunto de dados utilizado neste experimento é composto por informações estruturais e locacionais de imóveis residenciais, coletadas a partir de anúncios públicos. Os dados foram previamente tratados e normalizados para garantir compatibilidade com algoritmos baseados em distância, como o *k-NN Regressor*.


### Objetivo do Experimento

O objetivo deste experimento é avaliar o desempenho do algoritmo *k-NN Regressor* em um problema de regressão supervisionada, analisando o impacto do pré-processamento dos dados e da escolha do número de vizinhos (*k*) na qualidade das previsões.


### Organização do Notebook

O notebook está organizado da seguinte forma: inicialmente são importadas as bibliotecas necessárias e realizado o carregamento dos dados; em seguida, são aplicadas etapas de pré-processamento e normalização; posteriormente, o modelo *k-NN Regressor* é configurado e avaliado por meio de métricas de regressão; por fim, é apresentado um exemplo de previsão para uma nova instância.


In [ ]:
# ============================================================
# 0. Importação das bibliotecas
# ============================================================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
# ============================================================
# 1. Carregamento do Dataset via GitHub (reprodutível)
# ============================================================
url = "https://raw.githubusercontent.com/danielsantiago92/tcc-imoveis-knn/faba1dc076670b4bfd4cd53d6ae9370a6620f8fe/dados_imoveis_knn-r.csv"
df = pd.read_csv(url)

print("Pré-visualização dos dados:")
display(df.head())

Pré-visualização dos dados:


,Cidade,Bairro,Tipo_imovel,Area(m²),Quartos,Banheiros,Vagas,Preco(R$),Condominio(R$),Andar,Portaria 24h
0,Rio de Janeiro,Flamengo,Apto,263,4,6,1,2700000,2300,3,sim
1,Rio de Janeiro,Flamengo,Apto,113,3,2,1,1050000,1700,2,sim
2,Rio de Janeiro,Centro,Apto,16,1,1,0,130000,500,2,sim
3,Rio de Janeiro,Flamengo,Apto,66,2,2,0,670000,1200,2,sim
4,Rio de Janeiro,Leblon,Apto,257,4,4,3,5900000,5670,8,sim


In [ ]:
# ============================================================
# 2. Renomear colunas para padronizar
# ============================================================
df.columns = [
    "cidade", "bairro", "tipo_imovel", "area", "quartos", "banheiros",
    "vagas", "preco", "condominio", "andar", "portaria_24h"
]

In [ ]:

# ============================================================
# 3. Definir variáveis categóricas e numéricas
# ============================================================
cat_cols = ["cidade", "bairro", "tipo_imovel", "portaria_24h"]
num_cols = ["area", "quartos", "banheiros", "vagas", "condominio", "andar"]

In [ ]:
# ============================================================
# 4. Label Encoding das colunas categóricas
# ============================================================
encoders = {}
for col in cat_cols:
    enc = LabelEncoder()
    df[col] = enc.fit_transform(df[col])
    encoders[col] = enc

In [ ]:
# ============================================================
# 5. Separar X e y
# ============================================================
X = df[cat_cols + num_cols]
y = df["preco"]


In [ ]:
# ============================================================
# 6. Divisão em treino e teste
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# ============================================================
# 7. Normalização das variáveis numéricas
# ============================================================
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
# ============================================================
# 8. Busca automática do melhor k
# ============================================================
melhor_k = None
melhor_r2 = -999

for k in range(1, 41):  # testa valores de k entre 1 e 40
    model_temp = KNeighborsRegressor(n_neighbors=k, weights='distance')
    model_temp.fit(X_train, y_train)

    pred = model_temp.predict(X_test)
    r2_temp = r2_score(y_test, pred)

    if r2_temp > melhor_r2:
        melhor_r2 = r2_temp
        melhor_k = k

print(f"\nMelhor k encontrado: {melhor_k}")



Melhor k encontrado: 2


In [ ]:
# ============================================================
# 9. Treinar o modelo final com o melhor k
# ============================================================
model = KNeighborsRegressor(n_neighbors=melhor_k, weights='distance')
model.fit(X_train, y_train)

KNeighborsRegressor(n_neighbors=2, weights='distance')

In [ ]:
# ============================================================
# 10. Avaliação Final do Modelo
# ============================================================
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"\nCoeficiente de determinação (R²): {r2:.3f}")
print(f"Erro Absoluto Médio (MAE): R$ {mae:,.2f}")



Coeficiente de determinação (R²): 0.785
Erro Absoluto Médio (MAE): R$ 359,161.97


In [ ]:
# ============================================================
# 11. Previsão Manual (Exemplo Real)
# ============================================================
sample = {
    "cidade": ["Rio de Janeiro"],
    "bairro": ["Tijuca"],
    "tipo_imovel": ["Apto"],         # deve ser exatamente igual ao dataset
    "portaria_24h": ["sim"],         # valores devem existir no dataset
    "area": [85],
    "quartos": [2],
    "banheiros": [2],
    "vagas": [1],
    "condominio": [650],
    "andar": [5]
}

new_sample = pd.DataFrame(sample)


In [ ]:
# ============================================================
# 12. Codificar categorias
# ============================================================
for col in cat_cols:
    new_sample[col] = encoders[col].transform(new_sample[col])

In [ ]:
# ============================================================
# 13. Normalizar valores numéricos
# ============================================================
new_sample[num_cols] = scaler.transform(new_sample[num_cols])

In [ ]:
# ============================================================
# 14. Prever preço
# ============================================================
predicted_price = model.predict(new_sample)

print("\nPreço estimado para o imóvel:")
print(f"R$ {predicted_price[0]:,.2f}")


Preço estimado para o imóvel:
R$ 461,609.44
